In [ ]:
import os
import re
import json
import pickle
import random
import hashlib
import numpy as np

random.seed(42)
np.random.seed(42)

DATASET_ROOT = os.environ.get("DATASET_ROOT", "/content/dataset/Dataset")

MAX_FRAMES   = 110
JITTER_RATIO = 0.02    # relative to torso length (+-2%)
ROTATE_DEG   = 5       # max rotation in degrees
TEMP_SHIFT   = 1       # max frame shift
NUM_JOINTS   = 16      # 16 body joints left after cleanup

# Which subjects each class folder is read from (Train + Val) during training
ALL_TRAIN_SUBJECTS = ["Subject1", "Subject3", "Subject8"]
CLASS_FOLDER_SUBJECTS = {
    "jab":          ALL_TRAIN_SUBJECTS,
    "leadhook":     ALL_TRAIN_SUBJECTS,
    "cross":        ALL_TRAIN_SUBJECTS,
    "rearhook":     ALL_TRAIN_SUBJECTS,
    "leaduppercut": ALL_TRAIN_SUBJECTS,
    "rearuppercut": ALL_TRAIN_SUBJECTS,
    "nopunch":      ["Subject1"],              
}
TRAIN_SUBJECTS = sorted({s for subs in CLASS_FOLDER_SUBJECTS.values() for s in subs})
TEST_SUBJECTS  = ["Subject2", "Subject5", "Subject7"]    # never used in training

assert not set(TRAIN_SUBJECTS) & set(TEST_SUBJECTS), "Train/test subject overlap!"

CLASS_NAMES = ["jab", "leadhook", "cross_rearhook", "uppercut_nopunch"]
NUM_CLASSES = len(CLASS_NAMES)
NAME_W      = max(len(n) for n in CLASS_NAMES)    

# class folder name -> class label
# Adjust these depending on the model we want to train

FOLDER_LABEL = {
    "jab":          0,
    "leadhook":     1,
    "cross":        2,
    "rearhook":     2,
    "leaduppercut": 3,
    "rearuppercut": 3,
    "nopunch":      3,
}

# --- class balancing (training set only) ---
# ANCHOR classes (jab, leadhook) keep ALL their real clips; the smaller one is augmented up to the
# larger one, so the common target T = max(raw size of the anchors) (or BALANCE_TARGET_PER_CLASS).
# The other classes are brought to T as well: augmented up when smaller, randomly thinned when larger.
ANCHOR_CLASSES              = [0, 1]
BALANCE_TARGET_PER_CLASS    = None    # None = automatic (from the anchors), or an int; anchors are never cut
DOWNSAMPLE_NON_ANCHOR       = True    # False = never discard clips (then bigger classes keep their size)
FOLDER_ALLOCATION           = "equal"        # inside a class: "equal" share per folder, or "proportional" to raw count
SUBJECT_ALLOCATION          = "equal"        # inside a folder: "equal" share per subject, or "proportional" to raw count
MAX_AUG_PER_SEQ             = 40      # safety cap: max augmented copies of one original clip

SPLIT_TRAIN = ["train", "training"]
SPLIT_VAL   = ["val", "validation", "valid"]

MAX_IDS_PRINTED = 25   

# MediaPipe indexing after Z-removal and noise index removal
REMOVE_INDEXES = set([
    0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21,
    34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45
])

Folder Lookup

In [ ]:
def _norm(name):
    return re.sub(r'[^a-z0-9]', '', name.lower())

def find_child(parent, names):
    """Return the sub-directory of `parent` whose normalised name matches any of `names`."""
    if parent is None or not os.path.isdir(parent):
        return None
    wanted = {_norm(n) for n in names}
    for d in sorted(os.listdir(parent)):
        full = os.path.join(parent, d)
        if os.path.isdir(full) and _norm(d) in wanted:
            return full
    return None

def find_subject_dir(root, subject):
    """'Subject2' matches 'Subject2', 'subject2', 'Subject2(Test Subject)', ..."""
    if not os.path.isdir(root):
        return None
    for d in sorted(os.listdir(root)):
        full = os.path.join(root, d)
        if os.path.isdir(full) and _norm(d.split("(")[0]) == _norm(subject):
            return full
    return None

Loading and Cleaning

In [ ]:
def load_pickles_from_dir(directory, id_root=None):
    """Loads every .pkl under `directory`. Returns (sequences, ids); ids look like 'path/file.pkl#idx'."""
    sequences, ids = [], []
    if directory is None or not os.path.isdir(directory):
        return sequences, ids
    id_root = id_root or directory
    for root, dirs, files in os.walk(directory):
        dirs.sort()
        for file in sorted(files):
            if not file.endswith('.pkl'):
                continue
            path = os.path.join(root, file)
            rel = os.path.relpath(path, id_root)
            try:
                with open(path, 'rb') as f:
                    data = pickle.load(f)
            except Exception as e:
                print(f"Error loading {path}: {e}")
                continue
            if isinstance(data, list):
                sequences.extend(data)
                ids.extend([f"{rel}#{i}" for i in range(len(data))])
            else:
                sequences.append(data)
                ids.append(rel)
    return sequences, ids

def remove_z_coordinates(dataset):
    new_dataset = []
    for sequence in dataset:
        new_sequence = []
        for frame in sequence:
            new_sequence.append([value for i, value in enumerate(frame) if (i % 3) != 2])
        new_dataset.append(new_sequence)
    return new_dataset

def remove_noisy_coordinates(sequence):
    for data in sequence:
        for i in range(len(data)):
            data[i] = [value for index, value in enumerate(data[i]) if index not in REMOVE_INDEXES]
    return sequence

def preprocess_dataset(directory, with_ids=False, id_root=None):
    """Load -> drop empty sequences -> remove Z -> remove noisy joints."""
    seqs, ids = load_pickles_from_dir(directory, id_root=id_root)
    keep = [k for k, s in enumerate(seqs) if len(s) > 0]
    if len(keep) < len(seqs):
        print(f"  [warn] dropped {len(seqs) - len(keep)} empty sequence(s) in {directory}")
    seqs = [seqs[k] for k in keep]
    ids = [ids[k] for k in keep]
    if seqs:
        seqs = remove_z_coordinates(seqs)
        seqs = remove_noisy_coordinates(seqs)
    return (seqs, ids) if with_ids else seqs

Augmenting and Balancing

In [ ]:
def jitter_sequence_dynamic(seq, jitter_ratio=JITTER_RATIO):
    torso_lengths = []
    for frame in seq:
        shoulder = np.array([(frame[0] + frame[2]) / 2, (frame[1] + frame[3]) / 2])
        hip      = np.array([(frame[12] + frame[14]) / 2, (frame[13] + frame[15]) / 2])
        torso_lengths.append(np.linalg.norm(shoulder - hip))
    avg_torso = max(np.mean(torso_lengths), 1e-6)

    sample_jitter = np.random.uniform(0.005, jitter_ratio)
    noise = np.random.uniform(-sample_jitter, sample_jitter, seq.shape) * avg_torso
    return seq + noise

def rotate_sequence_dynamic(seq, max_deg=ROTATE_DEG):
    seq_rot = np.copy(seq)
    deg = 0
    while abs(deg) < 0.5:
        deg = np.random.uniform(-max_deg, max_deg)

    theta = np.radians(deg)
    cos_t, sin_t = np.cos(theta), np.sin(theta)

    frames, feature_dim = seq_rot.shape
    seq_rot = seq_rot.reshape((frames, NUM_JOINTS, 2))
    hip_center = (seq_rot[:, 6, :] + seq_rot[:, 7, :]) / 2

    for i in range(frames):
        for j in range(NUM_JOINTS):
            vec = seq_rot[i, j] - hip_center[i]
            rotated_x = vec[0] * cos_t - vec[1] * sin_t
            rotated_y = vec[0] * sin_t + vec[1] * cos_t
            seq_rot[i, j, 0] = hip_center[i, 0] + rotated_x
            seq_rot[i, j, 1] = hip_center[i, 1] + rotated_y

    return seq_rot.reshape((frames, feature_dim))

def temporal_shift_dynamic(seq, max_shift=TEMP_SHIFT):
    if len(seq) <= max_shift:
        return seq
    shift_val = np.random.choice([s for s in range(-max_shift, max_shift + 1) if s != 0])
    return np.roll(seq, shift=shift_val, axis=0)

def get_sequence_hash(seq):
    return hashlib.md5(np.round(seq, 5).tobytes()).hexdigest()

def augment_to_target(sequences, target_n):
    """Keep every (unique) original, then add augmented copies round-robin over the originals
    until `target_n` sequences exist (or MAX_AUG_PER_SEQ is hit for every original)."""
    arrs = [np.array(sq, dtype=np.float64) for sq in sequences]
    out, seen = [], set()
    for a in arrs:
        h = get_sequence_hash(a)
        if h not in seen:
            seen.add(h)
            out.append(a)

    need = target_n - len(out)
    if need <= 0 or not arrs:
        return out

    n = len(arrs)
    added_per_seq = [0] * n
    attempts, max_attempts, i = 0, need * 5 + 50, 0

    while need > 0 and attempts < max_attempts:
        idx = i % n
        i += 1
        if added_per_seq[idx] >= MAX_AUG_PER_SEQ:
            if all(c >= MAX_AUG_PER_SEQ for c in added_per_seq):
                break
            continue
        attempts += 1
        seq_aug = np.copy(arrs[idx])
        seq_aug = jitter_sequence_dynamic(seq_aug)
        seq_aug = rotate_sequence_dynamic(seq_aug)
        seq_aug = temporal_shift_dynamic(seq_aug)

        h = get_sequence_hash(seq_aug)
        if h not in seen:
            seen.add(h)
            out.append(seq_aug)
            added_per_seq[idx] += 1
            need -= 1

    return out

def compute_group_targets(groups):
    """Per (subject, folder) target counts so every class ends up the same size.
    Returns (targets, class_target, class_min_total)."""
    fractions, class_min_total = {}, {}
    for lbl in range(NUM_CLASSES):
        cg = [g for g in groups if g["label"] == lbl]
        folders = sorted({g["folder"] for g in cg})
        class_raw = sum(len(g["seqs"]) for g in cg)
        needed = 0.0
        for folder in folders:
            fg = [g for g in cg if g["folder"] == folder]
            folder_raw = sum(len(g["seqs"]) for g in fg)
            folder_frac = 1.0 / len(folders) if FOLDER_ALLOCATION == "equal" else folder_raw / class_raw
            for g in fg:
                subj_frac = 1.0 / len(fg) if SUBJECT_ALLOCATION == "equal" else len(g["seqs"]) / folder_raw
                frac = folder_frac * subj_frac
                fractions[(g["subject"], g["folder"])] = frac
                needed = max(needed, len(g["seqs"]) / frac)    # class total at which this group needs no cutting
        class_min_total[lbl] = needed

    anchors = [l for l in ANCHOR_CLASSES if l in class_min_total]
    if BALANCE_TARGET_PER_CLASS is None:
        t_global = max(class_min_total[l] for l in anchors)
    else:
        t_global = BALANCE_TARGET_PER_CLASS

    class_target, cut_allowed = {}, {}
    for lbl in class_min_total:
        cut_allowed[lbl] = (lbl not in anchors) and DOWNSAMPLE_NON_ANCHOR
        class_target[lbl] = int(round(t_global if cut_allowed[lbl] else max(t_global, class_min_total[lbl])))

    targets = {}
    for g in groups:
        k = (g["subject"], g["folder"])
        t = max(1, int(round(fractions[k] * class_target[g["label"]])))
        targets[k] = t if cut_allowed[g["label"]] else max(len(g["seqs"]), t)
    return targets, class_target, class_min_total

Loading Training/Validation

In [ ]:
def folders_for_subject(subject):
    return [f for f, subs in CLASS_FOLDER_SUBJECTS.items() if subject in subs]

print("=== LOADING TRAINING / VALIDATION DATA ===")
train_groups, val_groups = [], []

for subject in TRAIN_SUBJECTS:
    subj_dir = find_subject_dir(DATASET_ROOT, subject)
    if subj_dir is None:
        print(f"[!] {subject}: folder not found under {DATASET_ROOT} - skipped")
        continue

    train_dir = find_child(subj_dir, SPLIT_TRAIN)
    val_dir   = find_child(subj_dir, SPLIT_VAL)
    if train_dir is None:
        train_dir = subj_dir            
        print(f"[note] {subject}: no Train folder, reading class folders from the subject root")
    if val_dir is None:
        print(f"[note] {subject}: no Val folder - no validation data from this subject")

    for folder in folders_for_subject(subject):
        label = FOLDER_LABEL[folder]

        tr_dir = find_child(train_dir, [folder])
        tr = preprocess_dataset(tr_dir) if tr_dir else []
        if tr:
            train_groups.append({"subject": subject, "folder": folder, "label": label, "seqs": tr})
        else:
            print(f"[skip] {subject}/Train/{folder}: not found or empty")

        va = []
        if val_dir:
            v_dir = find_child(val_dir, [folder])
            va = preprocess_dataset(v_dir) if v_dir else []
        if va:
            val_groups.append({"subject": subject, "folder": folder, "label": label, "seqs": va})

for lbl in range(NUM_CLASSES):
    assert any(g["label"] == lbl for g in train_groups), \
        f"No training data found for class '{CLASS_NAMES[lbl]}' - check folder names / DATASET_ROOT"

# --- balance + augment training groups only ---
group_targets, class_target, class_min_total = compute_group_targets(train_groups)

thin_rng = np.random.RandomState(42)
x_train_raw, y_train_list = [], []
aug_counts, used_counts = {}, {}
for g in train_groups:
    k = (g["subject"], g["folder"])
    seqs, tgt = g["seqs"], group_targets[k]
    if tgt < len(seqs):    # only happens for non-anchor classes that have more data than the target
        keep = np.sort(thin_rng.choice(len(seqs), size=tgt, replace=False))
        seqs = [seqs[i] for i in keep]
    used_counts[k] = len(seqs)
    aug = augment_to_target(seqs, tgt)
    aug_counts[k] = len(aug)
    x_train_raw.extend(aug)
    y_train_list.extend([g["label"]] * len(aug))

perm = np.random.permutation(len(x_train_raw))
x_train_raw = [x_train_raw[i] for i in perm]
y_train = np.array(y_train_list, dtype=int)[perm]

# --- validation (raw, not augmented) ---
x_val_raw, y_val_list = [], []
for g in val_groups:
    x_val_raw.extend(g["seqs"])
    y_val_list.extend([g["label"]] * len(g["seqs"]))
if x_val_raw:
    vperm = np.random.permutation(len(x_val_raw))
    x_val_raw = [x_val_raw[i] for i in vperm]
    y_val = np.array(y_val_list, dtype=int)[vperm]
else:
    y_val = np.array([], dtype=int)

# --- coverage table ---
val_counts = {(g["subject"], g["folder"]): len(g["seqs"]) for g in val_groups}
raw_counts = {(g["subject"], g["folder"]): len(g["seqs"]) for g in train_groups}
print("\n=== DATA COVERAGE ===")
hdr = (f"{'Subject':<10} | {'Folder':<{NAME_W}} | {'Class':<{NAME_W}} | {'Train raw':>9} | {'Target':>7} | "
       f"{'Used raw':>8} | {'Final':>6} | {'Val raw':>7}")
print(hdr); print("-" * len(hdr))
for subject in TRAIN_SUBJECTS:
    for folder in folders_for_subject(subject):
        k = (subject, folder)
        print(f"{subject:<10} | {folder:<{NAME_W}} | {CLASS_NAMES[FOLDER_LABEL[folder]]:<{NAME_W}} | "
              f"{raw_counts.get(k, 0):>9} | {group_targets.get(k, 0):>7} | "
              f"{used_counts.get(k, 0):>8} | {aug_counts.get(k, 0):>6} | {val_counts.get(k, 0):>7}")
print("-" * len(hdr))
print("\nTraining sequences per class:")
final_counts = {}
for lbl in range(NUM_CLASSES):
    raw_total = sum(len(g["seqs"]) for g in train_groups if g["label"] == lbl)
    final_counts[lbl] = int((y_train == lbl).sum())
    print(f"  {CLASS_NAMES[lbl]:<{NAME_W}}: raw {raw_total:>5} -> target {class_target[lbl]:>5} -> final {final_counts[lbl]:>5}")
ratio = max(final_counts.values()) / max(1, min(final_counts.values()))
print(f"  largest/smallest class ratio: {ratio:.2f}")
if ratio > 1.2:
    print("  [warn] classes are not within 20% of each other - likely MAX_AUG_PER_SEQ was hit for the smallest group "
          "(raise it, lower BALANCE_TARGET_PER_CLASS, or use FOLDER_ALLOCATION='proportional').")
if len(y_val):
    print("Validation sequences per class:")
    for lbl in range(NUM_CLASSES):
        print(f"  {CLASS_NAMES[lbl]:<{NAME_W}}: {int((y_val == lbl).sum())}")
else:
    print("No validation data found for any training subject - training will monitor 'loss' instead of 'val_loss'.")

Translation and Scale Normalization

In [ ]:
import numpy as np

NUM_JOINTS = 16
FEATURE_DIM = NUM_JOINTS * 2

def get_hip_center(frame):
    lx, ly = frame[12], frame[13]
    rx, ry = frame[14], frame[15]
    return np.array([(lx + rx) / 2, (ly + ry) / 2])


def get_shoulder_center(frame):
    lx, ly = frame[0], frame[1]
    rx, ry = frame[2], frame[3]
    return np.array([(lx + rx) / 2, (ly + ry) / 2])


def scale_normalization(frame):
    hip_center = get_hip_center(frame)
    shoulder_center = get_shoulder_center(frame)
    torso = np.linalg.norm(shoulder_center - hip_center)
    if torso < 1e-6:
        torso = 1e-6
    scaled = [val / torso for val in frame]
    return scaled


def translation_normalization(frame):
    hip_center = get_hip_center(frame)
    normalized = []
    for i in range(0, len(frame), 2):
        joint_xy = np.array([frame[i], frame[i+1]])
        norm_xy = (joint_xy - hip_center).tolist()
        normalized.extend(norm_xy)
    return normalized


def temporal_cleaning(sequence, max_jump=0.2):
    if not sequence:
        return sequence

    cleaned_seq = [sequence[0]]
    for i in range(1, len(sequence)):
        prev_frame = cleaned_seq[-1]
        curr_frame = sequence[i]

        lw_prev = np.array([prev_frame[8], prev_frame[9]])
        lw_curr = np.array([curr_frame[8], curr_frame[9]])

        rw_prev = np.array([prev_frame[10], prev_frame[11]])
        rw_curr = np.array([curr_frame[10], curr_frame[11]])

        lw_dist = np.linalg.norm(lw_curr - lw_prev)
        rw_dist = np.linalg.norm(rw_curr - rw_prev)

        if lw_dist > max_jump or rw_dist > max_jump:
            cleaned_seq.append(prev_frame)
        else:
            cleaned_seq.append(curr_frame)
    return cleaned_seq


def preprocess_nested_sequence(sequence_data):
    preprocessed_sequence = []
    for punch in sequence_data:
        seq_translated = [translation_normalization(frame) for frame in punch]
        seq_scaled = [scale_normalization(frame) for frame in seq_translated]
        # seq_cleaned = temporal_cleaning(seq_scaled)
        preprocessed_sequence.append(seq_scaled)
    return preprocessed_sequence

x_train_preprocessed = preprocess_nested_sequence(x_train_aug)
x_val_preprocessed = preprocess_nested_sequence(x_val)

print("Preprocessing complete!")
print("Example frame shape:", np.array(x_train_preprocessed[0][0]).shape)


Feature Engineering

In [ ]:
import numpy as np

def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = abs((180.0 * radians) / np.pi)

    if angle > 180.0:
        angle = 360.0 - angle

    return angle


def extract_angles(frame):
    def get_xy(idx):
        return [frame[idx*2], frame[idx*2 + 1]]

    angle_l_arm = calculate_angle(get_xy(0), get_xy(2), get_xy(4))  # LS-LE-LW
    angle_r_arm = calculate_angle(get_xy(1), get_xy(3), get_xy(5))  # RS-RE-RW

    angle_l_torso = calculate_angle(get_xy(6), get_xy(0), get_xy(2))  # LH-LS-LE
    angle_r_torso = calculate_angle(get_xy(7), get_xy(1), get_xy(3))  # RH-RS-RE

    angle_l_leg = calculate_angle(get_xy(0), get_xy(6), get_xy(8))  # LS-LH-LK
    angle_r_leg = calculate_angle(get_xy(1), get_xy(7), get_xy(9))  # RS-RH-RK

    return [
        angle_l_arm, angle_r_arm,
        angle_l_torso, angle_r_torso,
        angle_l_leg, angle_r_leg
    ]


def extract_velocities(curr_frame, prev_frame):
    velocities = []

    # shoulders, elbows, wrists
    joint_ids = [0, 1, 2, 3, 4, 5]

    for j_id in joint_ids:
        vx = curr_frame[j_id*2] - prev_frame[j_id*2]
        vy = curr_frame[j_id*2 + 1] - prev_frame[j_id*2 + 1]
        velocities.extend([vx, vy])

    return velocities

def extract_wrist_speeds(velocities):
    # order: LS, RS, LE, RE, LW, RW
    lw_vx, lw_vy = velocities[8], velocities[9]
    rw_vx, rw_vy = velocities[10], velocities[11]

    lw_speed = np.sqrt(lw_vx**2 + lw_vy**2)
    rw_speed = np.sqrt(rw_vx**2 + rw_vy**2)

    return [lw_speed, rw_speed]


def wrist_to_shoulder_distance(frame):
    def get_xy(idx):
        return np.array([frame[idx*2], frame[idx*2 + 1]])

    l_wrist = get_xy(4)
    r_wrist = get_xy(5)
    l_shoulder = get_xy(0)
    r_shoulder = get_xy(1)

    dist_l = np.linalg.norm(l_wrist - l_shoulder)
    dist_r = np.linalg.norm(r_wrist - r_shoulder)

    return [dist_l, dist_r]

def lead_rear_asymmetry(frame):
    # Left wrist x - Right wrist x
    # Jab (lead=left): left wrist shoots forward → more negative x (closer to camera)
    # Cross (lead=left): right wrist shoots forward
    l_wrist_x = frame[4*2]      # index 8
    r_wrist_x = frame[5*2]      # index 10
    return [l_wrist_x - r_wrist_x]


def wrist_motion_angle(velocities):
    """Direction of travel (atan2(vy, vx), in radians) for each wrist,
    derived from the existing `velocities` array. velocities is ordered
    LS,RS,LE,RE,LW,RW (pairs of vx,vy) from extract_velocities, so LW sits
    at [8,9] and RW at [10,11]. Roughly 0/pi (horizontal) for hooks, roughly
    +-pi/2 (vertical) for uppercuts."""
    lw_angle = np.arctan2(velocities[9], velocities[8])
    rw_angle = np.arctan2(velocities[11], velocities[10])
    return [lw_angle, rw_angle]


def cumulative_wrist_displacement(curr_frame, frame0):
    """Vertical and horizontal displacement of each wrist relative to the
    punch's first frame (frame0), not the previous frame. Returns
    (vert_disp, horiz_disp), each [left, right]. At frame 0 itself, this is
    called with frame0 == curr_frame, which naturally yields all zeros --
    satisfying the "frame 0 should be 0" requirement without a separate
    hardcoded branch."""
    def get_xy(frame, idx):
        return frame[idx * 2], frame[idx * 2 + 1]

    lw_x0, lw_y0 = get_xy(frame0, 4)   # LW
    rw_x0, rw_y0 = get_xy(frame0, 5)   # RW
    lw_x, lw_y = get_xy(curr_frame, 4)
    rw_x, rw_y = get_xy(curr_frame, 5)

    vert_disp = [lw_y - lw_y0, rw_y - rw_y0]
    horiz_disp = [lw_x - lw_x0, rw_x - rw_x0]
    return vert_disp, horiz_disp


def vertical_horizontal_ratio(vert_disp, horiz_disp):
    """abs(vertical displacement) / abs(horizontal displacement) per wrist.
    High for uppercuts (mostly vertical travel), low for hooks (mostly
    horizontal travel). +1e-6 guards divide-by-zero when horizontal
    displacement is exactly 0 (including at frame 0, where both are 0 and
    this correctly evaluates to 0)."""
    lw_ratio = abs(vert_disp[0]) / (abs(horiz_disp[0]) + 1e-6)
    rw_ratio = abs(vert_disp[1]) / (abs(horiz_disp[1]) + 1e-6)
    return [lw_ratio, rw_ratio]


def elbow_shoulder_height(frame):
    """elbow_y - shoulder_y for each arm. Uppercuts tend to keep the elbow
    lower/tucked relative to the shoulder; hooks raise it toward shoulder
    height. Computed directly on the current frame (not a delta), same as
    extract_angles/wrist_to_shoulder_distance -- no frame-0 special case
    needed."""
    le_height = frame[2*2 + 1] - frame[0*2 + 1]  # LE_y - LS_y
    re_height = frame[3*2 + 1] - frame[1*2 + 1]  # RE_y - RS_y
    return [le_height, re_height]


def elbow_torso_offset(frame):
    """elbow_x - hip_center_x for each arm. Hooks tend to swing the elbow
    outward laterally more than uppercuts. Also a direct per-frame value,
    no frame-0 special case needed."""
    hip_center_x = (frame[6*2] + frame[7*2]) / 2.0  # (LHIP_x + RHIP_x) / 2
    le_offset = frame[2*2] - hip_center_x  # LE_x - hip_center_x
    re_offset = frame[3*2] - hip_center_x  # RE_x - hip_center_x
    return [le_offset, re_offset]


def build_feature_vector(curr_frame, prev_frame, frame0):

    angles = extract_angles(curr_frame)
    velocities = extract_velocities(curr_frame, prev_frame)
    speeds = extract_wrist_speeds(velocities)
    wrist_dist = wrist_to_shoulder_distance(curr_frame)
    asymmetry = lead_rear_asymmetry(curr_frame)

    motion_angle = wrist_motion_angle(velocities)
    vert_disp, horiz_disp = cumulative_wrist_displacement(curr_frame, frame0)
    vh_ratio = vertical_horizontal_ratio(vert_disp, horiz_disp)
    elbow_height = elbow_shoulder_height(curr_frame)
    elbow_offset = elbow_torso_offset(curr_frame)

    return np.concatenate([
        curr_frame,
        velocities,
        angles,
        speeds,
        wrist_dist,
        asymmetry,
        motion_angle,
        vert_disp,
        horiz_disp,
        vh_ratio,
        elbow_height,
        elbow_offset
    ])

def process_punch_sequence(punch):
    if len(punch) == 0:
        return []

    features = []

    zero_vel = [0.0] * 12
    zero_speed = [0.0, 0.0]
    asy = [0]

    frame0 = punch[0]  

    first_angles = extract_angles(punch[0])
    first_wrist_dist = wrist_to_shoulder_distance(punch[0])

    first_elbow_height = elbow_shoulder_height(punch[0])
    first_elbow_offset = elbow_torso_offset(punch[0])

    features.append(
        np.concatenate([
            punch[0],
            zero_vel,
            first_angles,
            zero_speed,
            first_wrist_dist,
            asy,
            first_motion_angle,
            first_vert_disp,
            first_horiz_disp,
            first_vh_ratio,
            first_elbow_height,
            first_elbow_offset
        ])
    )

    for i in range(1, len(punch)):
        fv = build_feature_vector(punch[i], punch[i - 1], frame0)
        features.append(fv)

    return features


_dummy_frame = [0.0] * 32
_dummy_fv = build_feature_vector(_dummy_frame, _dummy_frame, _dummy_frame)
FEATURE_DIM = len(_dummy_fv)
print(f"Per-frame feature vector dimensionality: {FEATURE_DIM}")

x_train_features_raw = [process_punch_sequence(seq) for seq in x_train_preprocessed]
x_val_features_raw = [process_punch_sequence(seq) for seq in x_val_preprocessed]


Padding

In [ ]:
def pad_sequence(seq, max_frames=110):
    seq = np.array(seq)
    frames, feature_dim = seq.shape

    if frames < max_frames:
        padding = np.zeros((max_frames - frames, feature_dim))
        seq = np.vstack([seq, padding])
    else:
        seq = seq[:max_frames]

    return seq

x_train_padded = np.array([pad_sequence(s, MAX_FRAMES) for s in x_train_features_raw])
x_val_padded = np.array([pad_sequence(s, MAX_FRAMES) for s in x_val_features_raw])

print("Final Training Shape:", x_train_padded.shape)
print("Final Validation Shape:", x_val_padded.shape)

Model Architecture and Training (Change depending on the model you are training)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Masking
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(42)

# Class weights: nopunch comes from a single subject, so counts are uneven across classes
cw = compute_class_weight(class_weight='balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print("Class weights:", {CLASS_NAMES[i]: round(w, 3) for i, w in class_weight.items()})
FEATURE_DIM = 67

model = Sequential([
    Input(shape=(MAX_FRAMES, FEATURE_DIM)),
    Masking(mask_value=0.0),
    LSTM(64, return_sequences=True),
    Dropout(0.4),
    LSTM(32, return_sequences=False),
    Dropout(0.4),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

have_val = len(x_val) > 0
monitor_metric = 'val_loss' if have_val else 'loss'

callbacks = [
    EarlyStopping(monitor=monitor_metric, patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor=monitor_metric, factor=0.5, patience=3, min_lr=1e-6),
]

fit_kwargs = dict(
    x=x_train, y=y_train,
    epochs=25, batch_size=32,
    class_weight=class_weight,
    callbacks=callbacks,
)
if have_val:
    fit_kwargs["validation_data"] = (x_val, y_val)

history = model.fit(**fit_kwargs)

if have_val:
    val_pred = np.argmax(model.predict(x_val, verbose=0), axis=1)
    print("\nValidation report (training subjects' Val folders):")
    print(classification_report(y_val, val_pred, labels=list(range(NUM_CLASSES)),
                                target_names=CLASS_NAMES, zero_division=0))